# Audit Aksesibilitas WCAG (axe-core) — website Dinas Sosial Provinsi

Notebook ini menjalankan **Langkah audit axe-core** sesuai Protokol Audit (bagian 6) di guidelines KTI:
- Satu browser (Chrome headless), satu viewport, satu timeout konsisten untuk semua website.
- Hanya mengaudit baris `page_sample` yang `page_url`-nya sudah terisi (baris kosong otomatis di-skip, tidak bikin error).
- Output kolom mengikuti skema `audit_findings` di Tabel Inti (bagian 5.1).

**Sebelum audit, notebook ini juga menerapkan kriteria inklusi (Tabel 8):**
Website yang tersisa `< 4` tipe halaman terisi akan **dikeluarkan sementara** dari audit (bukan dipaksa dianalisis dengan data tidak lengkap), dan dilaporkan terpisah supaya kamu tahu website mana yang perlu dilengkapi dulu.

**Cara pakai:**
1. Jalankan semua cell dari atas ke bawah.
2. Saat diminta, upload file `master_-_page_sample_v2.csv`.
3. Hasil akhir: `audit_findings.csv`, `page_sample_updated.csv` (dengan status terisi), `website_master.csv`, dan `excluded_websites.csv`.

## 1. Install dependencies (Chrome headless + Selenium)

In [ ]:
# Install Google Chrome (bukan paket chromium-chromedriver apt yang sering mismatch di Colab)
!wget -q -O /tmp/google-chrome-stable.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!ls -la /tmp/google-chrome-stable.deb
!apt-get update -qq
!apt install -y /tmp/google-chrome-stable.deb
!apt --fix-broken install -y

import subprocess, shutil

chrome_path = shutil.which('google-chrome') or shutil.which('google-chrome-stable')
if chrome_path is None:
    raise RuntimeError(
        "google-chrome tidak ketemu setelah instalasi. Scroll ke atas, baca output "
        "'apt install' di cell ini untuk lihat pesan errornya (biasanya dependency yang gagal)."
    )

chrome_version = subprocess.run([chrome_path, '--version'], capture_output=True, text=True).stdout.strip()
print("Chrome path:", chrome_path)
print("Chrome version:", chrome_version)
# Catat versi ini di naskah metodologi (bagian 6.1) sebagai bagian dari "versi tool yang dipakai".
!pip install -q -U selenium
# Selenium >= 4.6 punya Selenium Manager bawaan yang otomatis download chromedriver
# yang cocok dengan versi Chrome ini, jadi tidak perlu setup chromedriver manual.

-rw-r--r-- 1 root root 139922812 Jul 29 21:59 /tmp/google-chrome-stable.deb
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'google-chrome-stable' instead of '/tmp/google-chrome-stable.deb'
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0 libatk1.0-0
  libatk1.0-data libatspi2.0-0 libvulkan1 libxcomposite1 libxtst6
  mesa-vulkan-drivers session-migration
The following NEW packages will be installed:
  at-spi2-core google-chrome-stable gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk1.0-0 libatk1.0-data libatspi2.0-0 libvulkan1
  libxcomposite1 libxtst6 mesa-vulkan-drivers session-migration
0 upgraded, 12 newly installed, 0 to remove and 131 not upgraded.
Need to get 11

## 2. Upload `master_-_page_sample_v2.csv`

In [ ]:
from google.colab import files
import pandas as pd
import io

uploaded = files.upload()
csv_name = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[csv_name]), dtype=str).fillna('')

print(f"Total baris: {len(df)}")
print(f"Total website: {df['website_id'].nunique()}")

Saving master - page_sample_v2.csv to master - page_sample_v2.csv
Total baris: 170
Total website: 34


## 3. Terapkan kriteria inklusi (Tabel 8) & filter baris kosong

- Website dengan **< 4 tipe halaman terisi** → dikeluarkan sementara (`excluded_websites.csv`), tidak diaudit.
- Baris dengan `page_url` kosong pada website yang **lolos** kriteria → di-skip individual (tidak bikin error, tinggal disusulkan nanti).

In [ ]:
MIN_PAGE_TYPES = 4

filled = df[df['page_url'].str.strip() != '']
website_type_counts = filled.groupby('website_id')['page_type'].nunique()

excluded_website_ids = website_type_counts[website_type_counts < MIN_PAGE_TYPES].index.tolist()
included_website_ids = website_type_counts[website_type_counts >= MIN_PAGE_TYPES].index.tolist()

print(f"Website dikeluarkan sementara (< {MIN_PAGE_TYPES} tipe halaman): {excluded_website_ids}")
print(f"Website lolos kriteria inklusi dan akan diaudit: {len(included_website_ids)} website")

df_excluded = df[df['website_id'].isin(excluded_website_ids)].copy()
df_excluded.to_csv('excluded_websites.csv', index=False)

df_to_audit = df[
    (df['website_id'].isin(included_website_ids)) &
    (df['page_url'].str.strip() != '')
].copy()

skipped_pending = df[
    (df['website_id'].isin(included_website_ids)) &
    (df['page_url'].str.strip() == '')
]

print(f"Baris yang akan diaudit sekarang: {len(df_to_audit)}")
print(f"Baris pending (page_url masih kosong, disusulkan nanti): {len(skipped_pending)}")
if len(skipped_pending):
    print(skipped_pending[['page_id', 'website_id', 'page_type']].to_string(index=False))

Website dikeluarkan sementara (< 4 tipe halaman): ['W26']
Website lolos kriteria inklusi dan akan diaudit: 33 website
Baris yang akan diaudit sekarang: 162
Baris pending (page_url masih kosong, disusulkan nanti): 3
page_id website_id        page_type
   P033        W07          Layanan
   P135        W27 Interaksi/Kontak
   P138        W28          Layanan


## 4. Setup driver (konfigurasi konsisten — catat di metodologi)

Viewport, timeout, dan versi browser di cell ini **harus sama** untuk seluruh audit (dan untuk audit ulang reproducibility di bagian 6.3 nanti).

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, WebDriverException

VIEWPORT_WIDTH = 1366
VIEWPORT_HEIGHT = 768
PAGE_LOAD_TIMEOUT_SEC = 30  # network idle/timeout konsisten (bagian 6.1)

def new_driver():
    options = Options()
    import shutil as _shutil
    options.binary_location = _shutil.which("google-chrome") or _shutil.which("google-chrome-stable")
    options.add_argument('--headless=new')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument(f'--window-size={VIEWPORT_WIDTH},{VIEWPORT_HEIGHT}')
    # Tidak perlu Service(path) manual - Selenium Manager otomatis
    # mengunduh chromedriver yang cocok dengan Chrome yang terpasang.
    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT_SEC)
    return driver

# Test cepat: pastikan driver bisa start sebelum lanjut ke audit penuh
_test_driver = new_driver()
_test_driver.get("https://example.com")
print("Driver OK, judul halaman test:", _test_driver.title)
_test_driver.quit()

print(f"Viewport: {VIEWPORT_WIDTH}x{VIEWPORT_HEIGHT}, timeout: {PAGE_LOAD_TIMEOUT_SEC}s")

Driver OK, judul halaman test: Example Domain
Viewport: 1366x768, timeout: 30s


## 5. Inject axe-core & fungsi parsing WCAG SC

In [ ]:
import requests

AXE_CORE_URL = "https://cdnjs.cloudflare.com/ajax/libs/axe-core/4.9.1/axe.min.js"
AXE_SOURCE = requests.get(AXE_CORE_URL, timeout=20).text
print(f"axe-core source loaded: {len(AXE_SOURCE)} chars (versi 4.9.1 — catat di metodologi)")

def parse_wcag_sc(tags):
    """Ubah tag axe-core seperti 'wcag143' -> '1.4.3'.
    Aturan: digit pertama = principle, digit kedua = guideline (selalu 1 digit di WCAG),
    sisanya = nomor SC (bisa 1-2 digit)."""
    scs = []
    for t in tags:
        if t.startswith('wcag') and t[4:].isdigit() and len(t) > 4:
            digits = t[4:]
            if len(digits) >= 3:
                principle, guideline, sc = digits[0], digits[1], digits[2:]
                scs.append(f"{principle}.{guideline}.{sc}")
    return ";".join(sorted(set(scs)))

def parse_wcag_level(tags):
    if any(t in tags for t in ('wcag2aa', 'wcag21aa', 'wcag22aa')):
        return 'AA'
    if any(t in tags for t in ('wcag2a', 'wcag21a', 'wcag22a')):
        return 'A'
    return ''

AXE_RUN_SCRIPT = """
var callback = arguments[arguments.length - 1];
axe.run(document, {resultTypes: ['violations']}).then(function(results){
    callback(results);
}).catch(function(err){
    callback({error: String(err)});
});
"""

axe-core source loaded: 555031 chars (versi 4.9.1 — catat di metodologi)


## 6. Jalankan audit per halaman

In [ ]:
import time
import hashlib
import re
from datetime import datetime, timezone

def normalize_selector(selector_list):
    sel = " ".join(selector_list) if isinstance(selector_list, list) else str(selector_list)
    return re.sub(r'\s+', ' ', sel.strip().lower())

def make_signature(website_id, page_id, rule_id, normalized_selector):
    raw = f"{website_id}|{page_id}|{rule_id}|{normalized_selector}"
    return hashlib.sha256(raw.encode('utf-8')).hexdigest()[:16]

findings = []
status_updates = []  # {page_id, http_status, render_status, tested_at}

driver = new_driver()

for i, row in df_to_audit.iterrows():
    page_id = row['page_id']
    website_id = row['website_id']
    url = row['page_url'].strip()
    tested_at = datetime.now(timezone.utc).isoformat()

    render_status = 'unknown'
    http_status = ''

    try:
        driver.get(url)
        time.sleep(2)  # jeda singkat setelah load, konsisten untuk semua halaman
        render_status = 'rendered'
    except TimeoutException:
        render_status = 'timeout'
    except WebDriverException as e:
        render_status = f'error: {str(e)[:100]}'

    if render_status != 'rendered':
        status_updates.append({
            'page_id': page_id, 'http_status': http_status,
            'render_status': render_status, 'tested_at': tested_at
        })
        print(f"[SKIP] {page_id} ({website_id}) - {render_status}: {url}")
        continue

    try:
        driver.execute_script(AXE_SOURCE)
        results = driver.execute_async_script(AXE_RUN_SCRIPT)
    except WebDriverException as e:
        status_updates.append({
            'page_id': page_id, 'http_status': http_status,
            'render_status': f'axe_error: {str(e)[:100]}', 'tested_at': tested_at
        })
        print(f"[AXE ERROR] {page_id}: {e}")
        continue

    if isinstance(results, dict) and results.get('error'):
        status_updates.append({
            'page_id': page_id, 'http_status': http_status,
            'render_status': f'axe_error: {results["error"][:100]}', 'tested_at': tested_at
        })
        continue

    violations = results.get('violations', [])
    for v in violations:
        rule_id = v.get('id', '')
        tags = v.get('tags', [])
        wcag_sc = parse_wcag_sc(tags)
        wcag_level = parse_wcag_level(tags)
        impact_label = v.get('impact', '') or ''
        nodes = v.get('nodes', [])

        for node in nodes:
            target = node.get('target', [])
            selector = normalize_selector(target)
            signature = make_signature(website_id, page_id, rule_id, selector)
            findings.append({
                'finding_id': f"F{len(findings)+1:06d}",
                'page_id': page_id,
                'website_id': website_id,
                'rule_id': rule_id,
                'wcag_sc': wcag_sc,
                'wcag_level': wcag_level,
                'impact_label': impact_label,
                'affected_node_count': len(nodes),
                'selector': selector,
                'finding_signature': signature,
                'validation_status': 'auto-detected',
            })

    status_updates.append({
        'page_id': page_id, 'http_status': http_status,
        'render_status': 'rendered', 'tested_at': tested_at
    })
    print(f"[OK] {page_id} ({website_id}) - {len(violations)} rule gagal, {sum(len(v.get('nodes', [])) for v in violations)} node terdampak")

driver.quit()
print(f"\\nSelesai. Total temuan (per-node): {len(findings)}")

[OK] P001 (W01) - 7 rule gagal, 71 node terdampak
[OK] P002 (W01) - 6 rule gagal, 62 node terdampak
[OK] P003 (W01) - 6 rule gagal, 62 node terdampak
[OK] P004 (W01) - 6 rule gagal, 51 node terdampak
[OK] P005 (W01) - 6 rule gagal, 58 node terdampak
[OK] P006 (W02) - 5 rule gagal, 29 node terdampak
[OK] P007 (W02) - 5 rule gagal, 8 node terdampak
[OK] P008 (W02) - 7 rule gagal, 10 node terdampak
[OK] P009 (W02) - 6 rule gagal, 13 node terdampak
[OK] P010 (W02) - 6 rule gagal, 9 node terdampak
[OK] P011 (W03) - 4 rule gagal, 41 node terdampak
[OK] P012 (W03) - 2 rule gagal, 8 node terdampak
[OK] P013 (W03) - 2 rule gagal, 3 node terdampak
[OK] P014 (W03) - 4 rule gagal, 15 node terdampak
[OK] P015 (W03) - 4 rule gagal, 4 node terdampak
[OK] P016 (W04) - 5 rule gagal, 17 node terdampak
[OK] P017 (W04) - 4 rule gagal, 12 node terdampak
[OK] P018 (W04) - 4 rule gagal, 12 node terdampak
[OK] P019 (W04) - 5 rule gagal, 19 node terdampak
[OK] P020 (W04) - 5 rule gagal, 14 node terdampak
[OK] 

## 7. Simpan `audit_findings.csv`

In [ ]:
audit_findings_df = pd.DataFrame(findings, columns=[
    'finding_id', 'page_id', 'website_id', 'rule_id', 'wcag_sc', 'wcag_level',
    'impact_label', 'affected_node_count', 'selector', 'finding_signature', 'validation_status'
])
audit_findings_df.to_csv('audit_findings.csv', index=False)
print(f"Tersimpan: audit_findings.csv ({len(audit_findings_df)} baris)")
audit_findings_df.head()

Tersimpan: audit_findings.csv (4062 baris)


,finding_id,page_id,website_id,rule_id,wcag_sc,wcag_level,impact_label,affected_node_count,selector,finding_signature,validation_status
0,F000001,P001,W01,color-contrast,1.4.3,AA,serious,16,"cms\:widgets[name=""widget_home_kiri_atas""] > c...",53fb0c2c9ed6239a,auto-detected
1,F000002,P001,W01,color-contrast,1.4.3,AA,serious,16,"cms\:widgets[name=""widget_home_kiri_atas""] > c...",74547ca450c0b30c,auto-detected
2,F000003,P001,W01,color-contrast,1.4.3,AA,serious,16,"cms\:widgets[name=""widget_home_kiri_atas""] > c...",7c7963121775268c,auto-detected
3,F000004,P001,W01,color-contrast,1.4.3,AA,serious,16,"cms\:widgets[name=""widget_home_kiri_atas""] > c...",65fb7742bb30594a,auto-detected
4,F000005,P001,W01,color-contrast,1.4.3,AA,serious,16,"cms\:widgets[name=""widget_home_kanan_atas""] > ...",89fa035770fe9688,auto-detected


## 8. Update `page_sample` dengan status hasil audit

In [ ]:
status_df = pd.DataFrame(status_updates).set_index('page_id')

df_updated = df.copy().set_index('page_id')
for col in ['http_status', 'render_status', 'tested_at']:
    if col in status_df.columns:
        df_updated.loc[status_df.index, col] = status_df[col]
df_updated = df_updated.reset_index()

df_updated.to_csv('page_sample_updated.csv', index=False)
print("Tersimpan: page_sample_updated.csv")
df_updated[df_updated['page_id'].isin(status_df.index)][['page_id', 'website_id', 'render_status', 'tested_at']].head(10)

Tersimpan: page_sample_updated.csv


,page_id,website_id,render_status,tested_at
0,P001,W01,rendered,2026-08-02T06:20:49.517723+00:00
1,P002,W01,rendered,2026-08-02T06:21:01.588494+00:00
2,P003,W01,rendered,2026-08-02T06:21:05.287005+00:00
3,P004,W01,rendered,2026-08-02T06:21:08.470007+00:00
4,P005,W01,rendered,2026-08-02T06:21:11.641934+00:00
5,P006,W02,rendered,2026-08-02T06:21:21.744126+00:00
6,P007,W02,rendered,2026-08-02T06:21:34.849323+00:00
7,P008,W02,rendered,2026-08-02T06:21:39.647097+00:00
8,P009,W02,rendered,2026-08-02T06:21:44.016506+00:00
9,P010,W02,rendered,2026-08-02T06:21:47.897104+00:00


## 9. Bangun `website_master.csv` (skeleton)

In [ ]:
rendered_counts = df_updated[df_updated['render_status'] == 'rendered'].groupby('website_id').size()

website_master = pd.DataFrame({'website_id': included_website_ids})
website_master['total_pages_audited'] = website_master['website_id'].map(rendered_counts).fillna(0).astype(int)
website_master['audit_date'] = datetime.now(timezone.utc).date().isoformat()
website_master['crawl_status'] = website_master['total_pages_audited'].apply(
    lambda n: 'complete' if n >= MIN_PAGE_TYPES else 'incomplete'
)
# Kolom ini masih perlu dilengkapi manual: institution_name, domain, province_name, bps_region_code
for col in ['institution_name', 'domain', 'province_name', 'bps_region_code']:
    website_master[col] = ''

website_master.to_csv('website_master.csv', index=False)
print("Tersimpan: website_master.csv (lengkapi kolom institution_name/domain/province_name/bps_region_code manual)")
website_master.head()

Tersimpan: website_master.csv (lengkapi kolom institution_name/domain/province_name/bps_region_code manual)


,website_id,total_pages_audited,audit_date,crawl_status,institution_name,domain,province_name,bps_region_code
0,W01,5,2026-08-02,complete,,,,
1,W02,5,2026-08-02,complete,,,,
2,W03,5,2026-08-02,complete,,,,
3,W04,5,2026-08-02,complete,,,,
4,W05,5,2026-08-02,complete,,,,


## Catatan & langkah selanjutnya

- **axe-core hanya mendeteksi sebagian Success Criterion secara otomatis.** Sesuai bagian 6.2 guidelines, sebutkan hasil ini sebagai *"hambatan yang dapat dideteksi otomatis"*, bukan skor kepatuhan WCAG penuh.
- **Website yang dikeluarkan sementara** (lihat `excluded_websites.csv`) perlu dilengkapi dulu sampai ≥4 tipe halaman sebelum ikut audit.
- **Audit ulang reproducibility (bagian 6.3):** setelah ini, pilih minimal 10% halaman atau minimal 10 halaman untuk diaudit ulang dengan konfigurasi yang sama, lalu bandingkan `finding_signature` dan jumlah temuan.
- Setelah `audit_findings.csv` beres, langkah berikutnya (Minggu 5 di rencana kerja) adalah menyusun `sc_function_mapping` dan `bps_demography`.